# Email header feature engineering

Privacy rule: no subject, body, preview, or attachment content is loaded.

In [ ]:
PROJECT_ROOT = '..'
RANDOM_SEED = 365

In [ ]:
from pathlib import Path
ROOT = Path(PROJECT_ROOT).resolve()
DATA_ROOT = ROOT / 'data'
ARTIFACT_ROOT = ROOT / 'artifacts/models/current'

In [ ]:
from packages.ml.m365risk_ml.features import load_spamassassin, FEATURE_NAMES, FORBIDDEN_HEADERS
import matplotlib.pyplot as plt
frame = load_spamassassin(DATA_ROOT/'raw/spamassassin')
print(frame.shape); display(frame[FEATURE_NAMES].describe().T)
assert not any(name in frame.columns for name in FORBIDDEN_HEADERS)

In [ ]:
frame.groupby('label')[FEATURE_NAMES].mean().T.plot.bar(figsize=(13,5), title='Mean header features by class'); plt.tight_layout()

In [ ]:
import seaborn as sns
sns.heatmap(frame[FEATURE_NAMES + ['label']].corr(), cmap='vlag', center=0); plt.title('Feature correlations')

In [ ]:
import pyarrow.dataset as ds
enron_ds=ds.dataset(DATA_ROOT/'processed/email_metadata_v1',format='parquet',partitioning='hive')
enron=enron_ds.to_table(filter=ds.field('dataset')=='enron',columns=['sender_hash','recipient_hash','received_at','received_hops']).to_pandas()
routes=enron.groupby(['sender_hash','recipient_hash']).size().sort_values(ascending=False); display(routes.head(15).rename('messages'))
enron.set_index('received_at').resample('D').size().plot(figsize=(12,4),title='Enron normal-routing volume over time'); plt.tight_layout()
assert set(enron.sender_hash.dropna()).isdisjoint(set(frame.get('message_key', []))), 'identity leakage check'